In [ ]:
import pandas as pd 
import numpy as np 
import re 
import string 

import nltk

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 

pd.set_option('display.max_colwidth',None)

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
jira_df = pd.read_csv('../data/processed/jira_cleaned.csv')

In [4]:
jira_df.head()

,Summary,Priority,Summary_Length
0,Authentication failed when attempting Fetch command,High,52
1,OAuth token keeps expiring,Low,26
2,Unable to start SourceTree after updating from older version to 3.4.12,Medium,71
3,Keesp Asking for GitHub Login,Medium,29
4,Dark Theme - history changed files not displayed correctly,Low,58


In [5]:
jira_df['Summary'].sample(10, random_state=42)

29821         Not able to check out remote branch with pipe (vertical line) character
24319               'Keep staged changes' checkbox doesn't work when stashing changes
33766                                                      commit time in 24h format 
24445    Authentication failure - Keeps attempting to authenticate on single request.
5014                         Code signing timestamp is missing from SourceTree v3.2.6
10210                                                    On push not select baranches
12286                                 Impossible to access to the Remote Account page
18701                                       Калькулятор скидки: не правильный расчет.
23072                                                       添加大量文件后，SourceTree会长时间无响应
28147                                        Custom Actions broken in Beta-3.3.0-3678
Name: Summary, dtype: str

In [6]:
stop_words = stopwords.words('english')
print(len(stop_words))
print(stop_words[:30])

198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]


In [7]:
sample = jira_df.loc[0, "Summary"]

print(sample)

 Authentication failed when attempting Fetch command


In [8]:
def clean_text(text):
    """
    Cleans a Jira ticket summary for NLP processing.

    Steps:
    1. Lowercase
    2. Remove punctuation
    3. Tokenize
    4. Remove stopwords
    5. Lemmatize
    6. Join tokens back into a sentence
    """    
    return text



In [9]:
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [10]:
sample = jira_df.loc[5209,'Summary']
print(sample)

[3.4.9] Random crashes while starting a new gitflow feature 


In [11]:
clean = sample.translate(
    str.maketrans('','',string.punctuation)
)
print(clean)

349 Random crashes while starting a new gitflow feature 


In [12]:
lower_clean = clean.lower()
print(lower_clean)

349 random crashes while starting a new gitflow feature 


In [13]:
lemmatizer = WordNetLemmatizer() 

In [14]:
print(lemmatizer.lemmatize("running"))
print(lemmatizer.lemmatize("failed"))
print(lemmatizer.lemmatize("studies"))
print(lemmatizer.lemmatize("authentication")) 

print(lemmatizer.lemmatize("running",pos='v'))
print(lemmatizer.lemmatize("failed",pos='v')) 
print(lemmatizer.lemmatize("studies",pos='v')) 
print(lemmatizer.lemmatize("authentication",pos='v'))

running
failed
study
authentication
run
fail
study
authentication


In [15]:
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize("running"))
print(lemmatizer.lemmatize("running", pos="v"))

print(lemmatizer.lemmatize("failed"))
print(lemmatizer.lemmatize("failed", pos="v"))

print(lemmatizer.lemmatize("attempting"))
print(lemmatizer.lemmatize("attempting", pos="v")) 

running
run
failed
fail
attempting
attempt


In [16]:

lemmatizer = WordNetLemmatizer() 
def clean_text(text):
    """
    Cleans a Jira ticket summary for NLP processing.

    Steps:
    1. Lowercase
    2. Remove punctuation
    3. Tokenize
    4. Remove stopwords
    5. Lemmatize
    6. Join tokens back into a sentence
    """    
   #  print('Original Text: ')
   #  print(text)
    #lowerCase
    text = text.lower()
    # print('Lower Case Text: ')
    # print(text)
    
    #Remove punctuation
    text = text.translate(
        str.maketrans('','',string.punctuation)
    )
    # print('Punctuation Removed Text: ')
    # print(text)

    #tokenize
    tokens = text.split()
    # print('Tokens: ')
    # print(tokens)

    #remove stopwords
    stop_words = set(stopwords.words("english"))
    important_words = {'not','no','nor'}
    custom_stop_words = [word for word in stop_words if word not in important_words]
    tokens = [token for token in tokens if token not in custom_stop_words]
    # print('Stopwords Removed: ')
    # print(tokens)

    #Lemmatize 
    tokens = [lemmatizer.lemmatize(token,pos='v')
             for token in tokens]
    # print('Lemmatized Tokens: ')
    # print(tokens)

    text = " ".join(tokens)
    return text
cleaned = clean_text(sample)
# print('Preprocessed Text: ')
# print(cleaned)

The cleaned tokens were concatenated into a sentence using Python's join() function, as TF-IDF vectorization requires textual documents rather than token lists.

TF-iDF

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
tfidf = TfidfVectorizer()

In [19]:
jira_df['Clean_Summary'] = jira_df['Summary'].apply(clean_text)

In [20]:
jira_df.head()

,Summary,Priority,Summary_Length,Clean_Summary
0,Authentication failed when attempting Fetch command,High,52,authentication fail attempt fetch command
1,OAuth token keeps expiring,Low,26,oauth token keep expire
2,Unable to start SourceTree after updating from older version to 3.4.12,Medium,71,unable start sourcetree update older version 3412
3,Keesp Asking for GitHub Login,Medium,29,keesp ask github login
4,Dark Theme - history changed files not displayed correctly,Low,58,dark theme history change file not display correctly


In [21]:
jira_df = jira_df[jira_df["Clean_Summary"].str.strip() != ""].reset_index(drop=True)

In [22]:
tfidf.fit(jira_df["Clean_Summary"])

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [23]:
len(tfidf.vocabulary_)

1172

In [24]:
list(tfidf.vocabulary_.items())[:20]

[('authentication', 102),
 ('fail', 375),
 ('attempt', 98),
 ('fetch', 380),
 ('command', 210),
 ('oauth', 653),
 ('token', 983),
 ('keep', 539),
 ('expire', 364),
 ('unable', 1008),
 ('start', 923),
 ('sourcetree', 902),
 ('update', 1035),
 ('older', 659),
 ('version', 1077),
 ('3412', 26),
 ('keesp', 540),
 ('ask', 93),
 ('github', 429),
 ('login', 572)]

In [25]:
print(list(tfidf.vocabulary_.items())[:20])

[('authentication', 102), ('fail', 375), ('attempt', 98), ('fetch', 380), ('command', 210), ('oauth', 653), ('token', 983), ('keep', 539), ('expire', 364), ('unable', 1008), ('start', 923), ('sourcetree', 902), ('update', 1035), ('older', 659), ('version', 1077), ('3412', 26), ('keesp', 540), ('ask', 93), ('github', 429), ('login', 572)]


In [26]:
tfidf = TfidfVectorizer()

tfidf.fit(jira_df["Clean_Summary"])

print("Vocabulary Size:", len(tfidf.vocabulary_))

Vocabulary Size: 1172


In [27]:
list(tfidf.vocabulary_.items())[:20]

[('authentication', 102),
 ('fail', 375),
 ('attempt', 98),
 ('fetch', 380),
 ('command', 210),
 ('oauth', 653),
 ('token', 983),
 ('keep', 539),
 ('expire', 364),
 ('unable', 1008),
 ('start', 923),
 ('sourcetree', 902),
 ('update', 1035),
 ('older', 659),
 ('version', 1077),
 ('3412', 26),
 ('keesp', 540),
 ('ask', 93),
 ('github', 429),
 ('login', 572)]

In [28]:
print(tfidf.vocabulary_["authentication"])
print(tfidf.vocabulary_["oauth"])
print(tfidf.vocabulary_["fetch"])

102
653
380


In [29]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[102])

authentication


In [30]:
X = tfidf.transform(jira_df['Clean_Summary'])

In [31]:
X.shape

(41797, 1172)

In [32]:
type(X)

scipy.sparse._csr.csr_matrix

In [33]:
X[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 1172)>

In [34]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:20])

['0702' '075' '0xc0000142' '10' '100' '11' '19137' '2015' '20191010'
 '2019kb4533002' '2022' '231' '2311' '24h' '313' '326' '334' '336'
 '3363829' '337']


In [35]:
print(X[0].toarray())

[[0. 0. 0. ... 0. 0. 0.]]


In [36]:
print(X[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 1172)>
  Coords	Values
  (0, 98)	0.486966667811854
  (0, 102)	0.39486760294850654
  (0, 210)	0.5272969309998258
  (0, 375)	0.30292647883653895
  (0, 380)	0.486966667811854


In [37]:
row = X[0]

print(row.indices)
print(row.data)

[ 98 102 210 375 380]
[0.48696667 0.3948676  0.52729693 0.30292648 0.48696667]


In [38]:
clean_text("---")

''

In [39]:
jira_df.to_csv("../data/processed/jira_processed.csv", index=False)